In [1]:
import pandas as pd
import numpy as np
import requests
from pybaseball import statcast, playerid_lookup, pitching_stats
import warnings
warnings.filterwarnings('ignore')

print("All imports successful ✓")

All imports successful ✓


In [3]:
# 2025 pitching stats

from pybaseball import pitching_stats

# Pull 2025 FanGraphs pitching stats
print("Pulling 2025 pitching stats...")
pitching_2025 = pitching_stats(2025, 2025, qual=50)

print(f"Got {len(pitching_2025)} pitchers")
print(pitching_2025[['Name', 'Team', 'ERA', 'xERA', 'FIP', 'xFIP', 'SIERA']].head(10))

Pulling 2025 pitching stats...
Got 339 pitchers
                   Name Team   ERA  xERA   FIP  xFIP  SIERA
20         Tarik Skubal  DET  2.21  2.71  2.45  2.66   2.71
9           Paul Skenes  PIT  1.97  2.65  2.36  3.03   3.10
34   Cristopher Sanchez  PHI  2.50  3.02  2.55  2.77   3.02
39      Garrett Crochet  BOS  2.59  2.88  2.89  2.64   2.86
116          Logan Webb  SFG  3.22  3.58  2.60  2.78   3.14
185       Jesus Luzardo  PHI  3.92  3.33  2.90  3.25   3.40
33   Yoshinobu Yamamoto  LAD  2.49  2.72  2.94  3.05   3.32
66            Max Fried  NYY  2.86  3.37  3.07  3.41   3.60
28         Hunter Brown  HOU  2.43  3.15  3.14  3.19   3.39
156       Kevin Gausman  TOR  3.59  3.73  3.41  3.77   3.79


In [6]:
opening_day_games = [
    # March 25 - Opening Night (Netflix)
    {"game_id": 1,  "date": "2026-03-25", "away_team": "New York Yankees",     "home_team": "San Francisco Giants",  "away_starter": "Max Fried",           "home_starter": "Logan Webb",          "venue": "Oracle Park",          "city": "San Francisco", "lat": 37.7786, "lon": -122.3893},
    # March 26 - Opening Day
    {"game_id": 2,  "date": "2026-03-26", "away_team": "Pittsburgh Pirates",   "home_team": "New York Mets",         "away_starter": "Paul Skenes",         "home_starter": "Freddy Peralta",      "venue": "Citi Field",           "city": "New York",      "lat": 40.7571, "lon": -73.8458},
    {"game_id": 3,  "date": "2026-03-26", "away_team": "Chicago White Sox",    "home_team": "Milwaukee Brewers",     "away_starter": "Shane Smith",         "home_starter": "Jacob Misiorowski",   "venue": "American Family Field","city": "Milwaukee",     "lat": 43.0280, "lon": -87.9712},
    {"game_id": 4,  "date": "2026-03-26", "away_team": "Washington Nationals", "home_team": "Chicago Cubs",          "away_starter": "Cade Cavalli",        "home_starter": "Matthew Boyd",        "venue": "Wrigley Field",        "city": "Chicago",       "lat": 41.9484, "lon": -87.6553},
    {"game_id": 5,  "date": "2026-03-26", "away_team": "Minnesota Twins",      "home_team": "Baltimore Orioles",     "away_starter": "Joe Ryan",            "home_starter": "Trevor Rogers",       "venue": "Camden Yards",         "city": "Baltimore",     "lat": 39.2838, "lon": -76.6218},
    {"game_id": 6,  "date": "2026-03-26", "away_team": "Boston Red Sox",       "home_team": "Cincinnati Reds",       "away_starter": "Garrett Crochet",     "home_starter": "Andrew Abbott",       "venue": "Great American Ball Park","city": "Cincinnati", "lat": 39.0979, "lon": -84.5082},
    {"game_id": 7,  "date": "2026-03-26", "away_team": "Los Angeles Angels",   "home_team": "Houston Astros",        "away_starter": "Jose Soriano",        "home_starter": "Hunter Brown",        "venue": "Minute Maid Park",     "city": "Houston",       "lat": 29.7572, "lon": -95.3555},
    {"game_id": 8,  "date": "2026-03-26", "away_team": "Detroit Tigers",       "home_team": "San Diego Padres",      "away_starter": "Tarik Skubal",        "home_starter": "TBD",                 "venue": "Petco Park",           "city": "San Diego",     "lat": 32.7073, "lon": -117.1566},
    {"game_id": 9,  "date": "2026-03-26", "away_team": "Texas Rangers",        "home_team": "Philadelphia Phillies", "away_starter": "Nathan Eovaldi",      "home_starter": "Cristopher Sanchez",  "venue": "Citizens Bank Park",   "city": "Philadelphia",  "lat": 39.9057, "lon": -75.1665},
    {"game_id": 10, "date": "2026-03-26", "away_team": "Tampa Bay Rays",       "home_team": "St. Louis Cardinals",   "away_starter": "Drew Rasmussen",      "home_starter": "Matthew Liberatore",  "venue": "Busch Stadium",        "city": "St. Louis",     "lat": 38.6226, "lon": -90.1928},
    {"game_id": 11, "date": "2026-03-26", "away_team": "Arizona Diamondbacks", "home_team": "Los Angeles Dodgers",   "away_starter": "Zac Gallen",          "home_starter": "Yoshinobu Yamamoto",  "venue": "Dodger Stadium",       "city": "Los Angeles",   "lat": 34.0739, "lon": -118.2400},
    {"game_id": 12, "date": "2026-03-26", "away_team": "Cleveland Guardians",  "home_team": "Seattle Mariners",      "away_starter": "Tanner Bibee",        "home_starter": "Logan Gilbert",       "venue": "T-Mobile Park",        "city": "Seattle",       "lat": 47.5914, "lon": -122.3325},
    # March 27
    {"game_id": 13, "date": "2026-03-27", "away_team": "Oakland Athletics",    "home_team": "Toronto Blue Jays",     "away_starter": "Luis Severino",       "home_starter": "Kevin Gausman",       "venue": "Rogers Centre",        "city": "Toronto",       "lat": 43.6414, "lon": -79.3894},
    {"game_id": 14, "date": "2026-03-27", "away_team": "Colorado Rockies",     "home_team": "Miami Marlins",         "away_starter": "Kyle Freeland",       "home_starter": "Sandy Alcantara",     "venue": "loanDepot Park",       "city": "Miami",         "lat": 25.7781, "lon": -80.2197},
    {"game_id": 15, "date": "2026-03-27", "away_team": "Kansas City Royals",   "home_team": "Atlanta Braves",        "away_starter": "Cole Ragans",         "home_starter": "Chris Sale",          "venue": "Truist Park",          "city": "Atlanta",       "lat": 33.8908, "lon": -84.4678},
]

games_df = pd.DataFrame(opening_day_games)

# Save to data folder
games_df.to_csv('../data/opening_day_games.csv', index=False)

print(f"✓ Saved {len(games_df)} games with verified starters")
games_df[['date','away_team','home_team','away_starter','home_starter']]

✓ Saved 15 games with verified starters


,date,away_team,home_team,away_starter,home_starter
0,2026-03-25,New York Yankees,San Francisco Giants,Max Fried,Logan Webb
1,2026-03-26,Pittsburgh Pirates,New York Mets,Paul Skenes,Freddy Peralta
2,2026-03-26,Chicago White Sox,Milwaukee Brewers,Shane Smith,Jacob Misiorowski
3,2026-03-26,Washington Nationals,Chicago Cubs,Cade Cavalli,Matthew Boyd
4,2026-03-26,Minnesota Twins,Baltimore Orioles,Joe Ryan,Trevor Rogers
5,2026-03-26,Boston Red Sox,Cincinnati Reds,Garrett Crochet,Andrew Abbott
6,2026-03-26,Los Angeles Angels,Houston Astros,Jose Soriano,Hunter Brown
7,2026-03-26,Detroit Tigers,San Diego Padres,Tarik Skubal,TBD
8,2026-03-26,Texas Rangers,Philadelphia Phillies,Nathan Eovaldi,Cristopher Sanchez
9,2026-03-26,Tampa Bay Rays,St. Louis Cardinals,Drew Rasmussen,Matthew Liberatore


In [7]:
from pybaseball import statcast_pitcher, playerid_lookup
import time

# All confirmed starters (skip TBD)
starters = [
    "Max Fried", "Logan Webb", "Paul Skenes", "Freddy Peralta",
    "Shane Smith", "Jacob Misiorowski", "Cade Cavalli", "Matthew Boyd",
    "Joe Ryan", "Trevor Rogers", "Garrett Crochet", "Andrew Abbott",
    "Jose Soriano", "Hunter Brown", "Tarik Skubal", "Nathan Eovaldi",
    "Cristopher Sanchez", "Drew Rasmussen", "Matthew Liberatore",
    "Zac Gallen", "Yoshinobu Yamamoto", "Tanner Bibee", "Logan Gilbert",
    "Luis Severino", "Kevin Gausman", "Kyle Freeland", "Sandy Alcantara",
    "Cole Ragans", "Chris Sale"
]

# Look up player IDs
print("Looking up player IDs...")
player_ids = {}
for name in starters:
    parts = name.split()
    first, last = parts[0], parts[-1]
    try:
        result = playerid_lookup(last, first)
        if len(result) > 0:
            mlbam_id = result.iloc[0]['key_mlbam']
            player_ids[name] = int(mlbam_id)
            print(f"  ✓ {name}: {mlbam_id}")
        else:
            print(f"  ✗ {name}: NOT FOUND")
    except Exception as e:
        print(f"  ✗ {name}: ERROR - {e}")
    time.sleep(0.3)

print(f"\nFound {len(player_ids)} of {len(starters)} pitchers")

Looking up player IDs...
Gathering player lookup table. This may take a moment.
  ✓ Max Fried: 608331
  ✓ Logan Webb: 657277
  ✓ Paul Skenes: 694973
  ✓ Freddy Peralta: 642547
  ✓ Shane Smith: 681343
  ✓ Jacob Misiorowski: 694819
  ✓ Cade Cavalli: 676917
  ✗ Matthew Boyd: NOT FOUND
  ✓ Joe Ryan: 657746
  ✓ Trevor Rogers: 669432
  ✓ Garrett Crochet: 676979
  ✓ Andrew Abbott: 671096
  ✗ Jose Soriano: NOT FOUND
  ✓ Hunter Brown: 686613
  ✓ Tarik Skubal: 669373
  ✓ Nathan Eovaldi: 543135
  ✗ Cristopher Sanchez: NOT FOUND
  ✓ Drew Rasmussen: 656876
  ✓ Matthew Liberatore: 669461
  ✓ Zac Gallen: 668678
  ✓ Yoshinobu Yamamoto: 808967
  ✓ Tanner Bibee: 676440
  ✓ Logan Gilbert: 669302
  ✓ Luis Severino: 622663
  ✓ Kevin Gausman: 592332
  ✓ Kyle Freeland: 607536
  ✗ Sandy Alcantara: NOT FOUND
  ✓ Cole Ragans: 666142
  ✓ Chris Sale: 519242

Found 25 of 29 pitchers


In [8]:
# Manual fixes for lookup failures
# These IDs are from Baseball Savant / MLB.com

manual_ids = {
    "Matthew Boyd":       571946,
    "Jose Soriano":       666624,
    "Cristopher Sanchez": 671218,
    "Sandy Alcantara":    645261,
}

player_ids.update(manual_ids)
print(f"Total pitchers: {len(player_ids)}")
print("All starters accounted for ✓")

Total pitchers: 29
All starters accounted for ✓


In [9]:
# Pull 2025 Statcast data for all Opening Day starters
# This will take several minutes - grabbing a full season of pitch data

all_pitches = []

for name, mlbam_id in player_ids.items():
    try:
        print(f"Pulling {name}...")
        df = statcast_pitcher('2025-03-01', '2025-11-01', player_id=mlbam_id)
        if len(df) > 0:
            df['pitcher_name'] = name
            all_pitches.append(df)
            print(f"  ✓ {name}: {len(df)} pitches")
        else:
            print(f"  ✗ {name}: no data")
    except Exception as e:
        print(f"  ✗ {name}: ERROR - {e}")
    time.sleep(0.5)

# Combine into one dataframe
pitches_df = pd.concat(all_pitches, ignore_index=True)
print(f"\n✓ Total pitches: {len(pitches_df):,}")
print(f"✓ Pitchers: {pitches_df['pitcher_name'].nunique()}")

# Save raw data
pitches_df.to_parquet('../data/starter_pitches_2025.parquet', index=False)
print("✓ Saved to data/starter_pitches_2025.parquet")

Pulling Max Fried...
Gathering Player Data
  ✓ Max Fried: 3515 pitches
Pulling Logan Webb...
Gathering Player Data
  ✓ Logan Webb: 3498 pitches
Pulling Paul Skenes...
Gathering Player Data
  ✓ Paul Skenes: 3326 pitches
Pulling Freddy Peralta...
Gathering Player Data
  ✓ Freddy Peralta: 3568 pitches
Pulling Shane Smith...
Gathering Player Data
  ✓ Shane Smith: 2556 pitches
Pulling Jacob Misiorowski...
Gathering Player Data
  ✓ Jacob Misiorowski: 1466 pitches
Pulling Cade Cavalli...
Gathering Player Data
  ✓ Cade Cavalli: 790 pitches
Pulling Joe Ryan...
Gathering Player Data
  ✓ Joe Ryan: 3026 pitches
Pulling Trevor Rogers...
Gathering Player Data
  ✓ Trevor Rogers: 1627 pitches
Pulling Garrett Crochet...
Gathering Player Data
  ✓ Garrett Crochet: 3477 pitches
Pulling Andrew Abbott...
Gathering Player Data
  ✓ Andrew Abbott: 2775 pitches
Pulling Hunter Brown...
Gathering Player Data
  ✓ Hunter Brown: 3167 pitches
Pulling Tarik Skubal...
Gathering Player Data
  ✓ Tarik Skubal: 3404 pitche

In [10]:
# Sanity check - what columns and data quality do we have?
print("Shape:", pitches_df.shape)
print("\nKey columns available:")
key_cols = [
    'pitcher_name', 'pitch_type', 'pitch_name',
    'release_speed', 'pfx_x', 'pfx_z',
    'release_spin_rate', 'whiff_rate',
    'estimated_woba_using_speedangle',
    'delta_run_exp'
]
for col in key_cols:
    available = col in pitches_df.columns
    null_pct = pitches_df[col].isna().mean() * 100 if available else 100
    print(f"  {'✓' if available else '✗'} {col} — {null_pct:.1f}% null")

Shape: (72802, 119)

Key columns available:
  ✓ pitcher_name — 0.0% null
  ✓ pitch_type — 3.2% null
  ✓ pitch_name — 3.2% null
  ✓ release_speed — 3.2% null
  ✓ pfx_x — 3.2% null
  ✓ pfx_z — 3.2% null
  ✓ release_spin_rate — 3.4% null
  ✗ whiff_rate — 100.0% null
  ✓ estimated_woba_using_speedangle — 76.3% null
  ✓ delta_run_exp — 0.1% null


In [12]:
# Calculate pitch arsenal metrics from raw pitch data

# First, calculate whiff on each pitch (swing and miss)
pitches_df['is_swing'] = pitches_df['description'].isin([
    'swinging_strike', 'swinging_strike_blocked', 
    'foul', 'foul_tip', 'hit_into_play', 'hit_into_play_no_out'
])
pitches_df['is_whiff'] = pitches_df['description'].isin([
    'swinging_strike', 'swinging_strike_blocked', 'foul_tip'
])

# Build arsenal summary
arsenal = (
    pitches_df
    .groupby(['pitcher_name', 'pitch_type', 'pitch_name'])
    .agg(
        count        = ('pitch_type', 'count'),
        usage_pct    = ('pitch_type', 'count'),  # will normalize below
        avg_velo     = ('release_speed', 'mean'),
        avg_spin     = ('release_spin_rate', 'mean'),
        pfx_x        = ('pfx_x', 'mean'),        # horizontal movement (inches)
        pfx_z        = ('pfx_z', 'mean'),        # vertical movement (inches)
        whiffs       = ('is_whiff', 'sum'),
        swings       = ('is_swing', 'sum'),
        delta_run_exp= ('delta_run_exp', 'sum'),
    )
    .reset_index()
)

# Calculate whiff rate and usage %
arsenal['whiff_rate'] = (arsenal['whiffs'] / arsenal['swings']).fillna(0)
total_pitches = arsenal.groupby('pitcher_name')['count'].transform('sum')
arsenal['usage_pct'] = arsenal['count'] / total_pitches * 100

# Convert movement to inches
arsenal['pfx_x_in'] = arsenal['pfx_x'] * 12
arsenal['pfx_z_in'] = arsenal['pfx_z'] * 12

# Filter out rare pitch types (< 2% usage)
arsenal = arsenal[arsenal['usage_pct'] >= 2].copy()

print(f"✓ Arsenal built: {len(arsenal)} pitch type rows")
print(f"✓ Pitchers: {arsenal['pitcher_name'].nunique()}")
print()
print(arsenal[['pitcher_name','pitch_name','usage_pct','avg_velo','whiff_rate','pfx_x_in','pfx_z_in']].sort_values(['pitcher_name','usage_pct'], ascending=[True,False]).head(20).to_string(index=False))


✓ Arsenal built: 138 pitch type rows
✓ Pitchers: 27

  pitcher_name      pitch_name  usage_pct  avg_velo  whiff_rate   pfx_x_in   pfx_z_in
 Andrew Abbott 4-Seam Fastball  47.531532 92.721759    0.196774   9.004185  16.203639
 Andrew Abbott        Changeup  19.531532 84.803137    0.255973  15.128635  10.213506
 Andrew Abbott       Curveball  14.702703 80.958333    0.241573  -9.970000  -3.173235
 Andrew Abbott         Sweeper  14.090090 82.765985    0.341014 -11.258517   3.537698
 Andrew Abbott          Cutter   4.144144 88.648696    0.225352  -0.517565   8.450087
  Cade Cavalli   Knuckle Curve  30.203046 86.152941    0.392857   6.300504 -12.911092
  Cade Cavalli 4-Seam Fastball  29.568528 97.140773    0.258621  -4.355021  15.042232
  Cade Cavalli          Sinker  18.401015 96.876552    0.097222 -13.696552   7.838897
  Cade Cavalli        Changeup  13.578680 89.808411    0.418182 -17.697196   6.111028
  Cade Cavalli          Cutter   8.121827 93.521875    0.121212   0.470625   8.643750
 

In [13]:
# Save arsenal summary
arsenal.to_csv('../data/arsenal_summary.csv', index=False)
print("✓ Saved arsenal_summary.csv")

# Quick look at the most devastating pitch types by whiff rate
print("\nTop 10 nastiest pitches (min 50 thrown):")
top_whiff = (
    arsenal[arsenal['count'] >= 50]
    .sort_values('whiff_rate', ascending=False)
    [['pitcher_name','pitch_name','usage_pct','avg_velo','whiff_rate']]
    .head(10)
)
top_whiff['usage_pct'] = top_whiff['usage_pct'].round(1)
top_whiff['avg_velo'] = top_whiff['avg_velo'].round(1)
top_whiff['whiff_rate'] = top_whiff['whiff_rate'].round(3)
print(top_whiff.to_string(index=False))

✓ Saved arsenal_summary.csv

Top 10 nastiest pitches (min 50 thrown):
   pitcher_name   pitch_name  usage_pct  avg_velo  whiff_rate
 Freddy Peralta       Slider       10.9      83.4       0.497
  Logan Gilbert Split-Finger       19.9      81.7       0.497
   Tarik Skubal     Changeup       30.3      88.1       0.487
Garrett Crochet     Changeup        2.5      88.7       0.485
    Shane Smith    Curveball       14.4      82.1       0.481
    Cole Ragans       Slider       14.8      85.0       0.460
    Cole Ragans     Changeup       19.1      84.5       0.459
 Nathan Eovaldi    Curveball       19.7      75.8       0.448
   Matthew Boyd Split-Finger       27.4      86.9       0.443
    Paul Skenes     Changeup       10.8      88.7       0.442


## ⚾ Arsenal Fingerprinting. What Each Starter Actually Throws

Every pitcher has a signature. Some live off one dominant pitch, others mix five offerings to keep hitters off balance. 
Using 72,802 Statcast pitches from the 2025 season, I mapped the complete arsenal for all 27 confirmed 
Opening Day starters — velocity, movement profile, and whiff rate for every pitch type they threw.

**The most interesting finding before we even get to matchups:**
Paul Skenes — the most hyped arm in baseball — generates more swings and misses with his *changeup* (44.2% whiff rate) 
than his legendary fastball. Tarik Skubal, the reigning AL Cy Young winner, same story. 
The fastball sets the table. The offspeed pitch eats.

## 📊 The Numbers Behind the Names: FanGraphs Pitching Stats

ERA tells you what happened. The advanced metrics tell you what should have happened.

A quick glossary before we dig in:

**ERA**: Earned Run Average. The classic. Runs allowed per 9 innings. Everyone knows this one.

**xERA**: Expected ERA. Strips out defense and luck, based purely on the quality of contact 
the pitcher allowed. A pitcher with a 4.50 ERA and a 3.20 xERA was probably a lot better than his record shows.

**FIP**: Fielding Independent Pitching. Only counts strikeouts, walks, and home runs. 
Everything the pitcher controls directly, nothing else.

**xFIP**: Like FIP, but also normalizes home run rate. Because sometimes pitchers just get 
unlucky with the long ball.

**SIERA**: Skill-Interactive ERA. The most complex of the group. Accounts for how 
strikeout and walk rates interact with each other. Generally the most predictive of future performance.

The gap between ERA and these metrics is where the interesting stories live. 
That gap is what we are hunting.

In [53]:
from pybaseball import pitching_stats

print("Pulling 2025 FanGraphs pitching stats...")
fg_2025 = pitching_stats(2025, 2025, qual=20)

# Keep only our Opening Day starters
starter_names = list(player_ids.keys())
fg_starters = fg_2025[fg_2025['Name'].isin(starter_names)].copy()

print(f"Matched {len(fg_starters)} of {len(starter_names)} starters")
print()

cols = ['Name', 'Team', 'ERA', 'xERA', 'FIP', 'xFIP', 'SIERA', 'IP', 'K_pct', 'BB_pct', 'WAR']
# Only show cols that exist
cols = [c for c in cols if c in fg_starters.columns]
print(fg_starters[cols].sort_values('xERA').to_string(index=False))

Pulling 2025 FanGraphs pitching stats...
Matched 29 of 29 starters

              Name Team  ERA  xERA  FIP  xFIP  SIERA    IP  WAR
       Paul Skenes  PIT 1.97  2.65 2.36  3.03   3.10 187.2  6.5
       Cole Ragans  KCR 4.67  2.67 2.50  2.45   2.52  61.2  2.1
      Tarik Skubal  DET 2.21  2.71 2.45  2.66   2.71 195.1  6.6
Yoshinobu Yamamoto  LAD 2.49  2.72 2.94  3.05   3.32 173.2  5.0
        Chris Sale  ATL 2.58  2.85 2.67  2.98   2.88 125.2  3.6
   Garrett Crochet  BOS 2.59  2.88 2.89  2.64   2.86 205.1  5.8
    Nathan Eovaldi  TEX 1.73  3.02 2.80  3.03   3.17 130.0  3.7
Cristopher Sanchez  PHI 2.50  3.02 2.55  2.77   3.02 202.0  6.4
     Logan Gilbert  SEA 3.44  3.08 3.35  2.95   2.86 131.0  2.6
      Hunter Brown  HOU 2.43  3.15 3.14  3.19   3.39 185.1  4.6
         Max Fried  NYY 2.86  3.37 3.07  3.41   3.60 195.1  4.8
     Trevor Rogers  BAL 1.81  3.40 2.82  3.64   3.75 109.2  3.3
 Jacob Misiorowski  MIL 4.36  3.40 3.62  3.66   3.56  66.0  1.2
    Freddy Peralta  MIL 2.70  3.42 3

In [15]:
fg_starters.to_csv('../data/fg_stats_2025.csv', index=False)
print("✓ Saved fg_stats_2025.csv")

✓ Saved fg_stats_2025.csv


In [58]:
import plotly.graph_objects as go

fg_plot = fg_starters[['Name', 'ERA', 'xERA', 'IP', 'WAR']].copy()

fg_plot['gap'] = fg_plot['ERA'] - fg_plot['xERA']
fg_plot['verdict'] = fg_plot['gap'].apply(
    lambda x: 'Fade (got lucky)' if x < -0.5
    else 'Ride (got unlucky)' if x > 0.5
    else 'True to form'
)

colors = {
    'Fade (got lucky)': '#dc2626',
    'Ride (got unlucky)': '#16a34a',
    'True to form': '#94a3b8'
}

fig = go.Figure()

for verdict, group in fg_plot.groupby('verdict'):
    fig.add_trace(go.Scatter(
        x=group['ERA'],
        y=group['xERA'],
        mode='markers+text',
        name=verdict,
        text=group['Name'],
        textposition='top center',
        textfont=dict(size=9, color='#1e293b'),
        marker=dict(
            size=group['IP'] / 15,
            color=colors[verdict],
            opacity=0.85,
            line=dict(width=1, color='#1e293b')
        ),
        hovertemplate='<b>%{text}</b><br>ERA: %{x}<br>xERA: %{y}<extra></extra>'
    ))

min_val, max_val = 1.5, 5.5
fig.add_trace(go.Scatter(
    x=[min_val, max_val],
    y=[min_val, max_val],
    mode='lines',
    line=dict(color='#94a3b8', dash='dash', width=1.5),
    showlegend=False,
    hoverinfo='skip'
))

fig.update_layout(
    title=dict(
        text='Fade or Ride — 2025 ERA vs xERA<br><sup>Bubble size = innings pitched. Above the line = got lucky. Below = got unlucky.</sup>',
        font=dict(color='#0f172a', size=18),
        x=0.05
    ),
    paper_bgcolor='white',
    plot_bgcolor='#f8fafc',
    font=dict(color='#1e293b', family='Arial'),
    xaxis=dict(
        title=dict(text='ERA (what happened)', font=dict(color='#1e293b', size=13)),
        gridcolor='#e2e8f0',
        zerolinecolor='#cbd5e1',
        tickfont=dict(color='#1e293b'),
        range=[min_val, max_val]
    ),
    yaxis=dict(
        title=dict(text='xERA (what should have happened)', font=dict(color='#1e293b', size=13)),
        gridcolor='#e2e8f0',
        zerolinecolor='#cbd5e1',
        tickfont=dict(color='#1e293b'),
        range=[min_val, max_val]
    ),
    legend=dict(
        bgcolor='white',
        bordercolor='#e2e8f0',
        borderwidth=1,
        font=dict(color='#1e293b')
    ),
    height=950,
    width=1400

)

fig.show()

### What the Numbers Are Already Telling Us

Before we build a single model, the data is already whispering.

Nathan Eovaldi posted a 1.73 ERA last season. His xERA was 3.02. That is a huge gap, 
and it means his results were heavily influenced by factors outside his control. 
Sequencing, defense, strand rate. The underlying stuff was good, not historic.

Trevor Rogers, same story. Cole Ragans is the mirror image. A 4.67 ERA that made him 
look like a back-of-the-rotation arm, hiding a 2.67 xERA that says he was elite. 
The Royals knew. Now you do too.

In [20]:
# Park factors sourced from Baseball Savant 2025 actuals + FantasyPros 2026 analysis
# Scale: 100 = league average, >100 hitter friendly, <100 pitcher friendly

park_factors = {
    "Oracle Park":              94,   # run suppressing
    "Citi Field":               95,   # run suppressing
    "American Family Field":    100,
    "Wrigley Field":            94,   # run suppressing
    "Camden Yards":             107,  # renovated 2025, now a launching pad
    "Great American Ball Park": 112,  # consistently top HR park
    "Minute Maid Park":         100,
    "Petco Park":               93,   # run suppressing
    "Citizens Bank Park":       105,
    "Busch Stadium":            97,
    "Dodger Stadium":           97,
    "T-Mobile Park":            92,   # most pitcher friendly park in MLB
    "Rogers Centre":            102,
    "loanDepot Park":           103,  # run amplifying per 2026 analysis
    "Truist Park":              99,
}

pf_df = pd.DataFrame(list(park_factors.items()), columns=['venue', 'park_factor'])
games_df = games_df.merge(pf_df, on='venue', how='left')

games_df.to_csv('../data/opening_day_games.csv', index=False)
print("✓ Park factors merged and saved")
print()
print(games_df[['venue','park_factor']].drop_duplicates().sort_values('park_factor', ascending=False).to_string(index=False))

✓ Park factors merged and saved

                   venue  park_factor
Great American Ball Park          112
            Camden Yards          107
      Citizens Bank Park          105
          loanDepot Park          103
           Rogers Centre          102
        Minute Maid Park          100
   American Family Field          100
             Truist Park           99
           Busch Stadium           97
          Dodger Stadium           97
              Citi Field           95
           Wrigley Field           94
             Oracle Park           94
              Petco Park           93
           T-Mobile Park           92


### The Park is a Silent Participant

Think of park factor as a multiplier on run scoring, centered at 100.

**100** means the park is perfectly neutral. A run scored there is exactly what you would expect 
in an average MLB environment.

**112 (Great American Ball Park)** means 12% more runs score there than in an average park. 
Short fences, warm Cincinnati air, hitter friendly dimensions. Pitchers hate it.

**92 (T-Mobile Park)** means 8% fewer runs score there than average. Cavernous outfield, 
cool marine air off Puget Sound, deep gaps. Hitters hate it.

So when we build the run environment model for each Opening Day game, we are not just looking 
at the two pitchers. We are asking how many runs should we expect in this specific place. 
A Skubal vs Peralta matchup at T-Mobile projects very differently than the same two pitchers 
at Great American Ball Park.

The park is a silent third participant in every game. These numbers let us account for that.

## 🏟️ Knowing Who Is In The Box: Lineup Quality Scores

A pitcher does not face a team. He faces nine individuals, each with their own tendencies, 
weaknesses, and strengths. 

To model run environment properly we need to know how dangerous each Opening Day lineup 
actually is. We will use wOBA, xwOBA, strikeout rate, walk rate, hard hit rate, and barrel 
rate to build a single Lineup Quality Score for each team.

A quick glossary:

**wOBA**: Weighted On Base Average. Assigns a value to every offensive outcome based on 
how many runs it actually produces. A single is worth less than a double, a walk less than 
a single. The most complete single number for offensive production.

**xwOBA**: Expected wOBA. Same concept but based on the quality of contact, not the result. 
Strips out luck and defense. A screaming liner that gets caught is still a good at bat.

**wRC+**: Weighted Runs Created Plus. Park and league adjusted offensive value. 
100 is perfectly average. 150 means 50% better than average. Dead simple to interpret.

**Barrel%**: The percentage of batted balls hit with the ideal combination of exit velocity 
and launch angle for maximum damage. If a ball is barreled, bad things happen to pitchers.

In [45]:
# Projected Opening Day lineups from MLB.com beat writers (March 23, 2026)
# Source: https://www.mlb.com/news/projected-lineups-rotations-for-every-2026-mlb-team

lineups = {
    "New York Yankees": [
        "Trent Grisham", "Aaron Judge", "Cody Bellinger", "Ben Rice",
        "Giancarlo Stanton", "Jazz Chisholm Jr.", "Ryan McMahon",
        "Jose Caballero", "Austin Wells"
    ],
    "San Francisco Giants": [
        "Willy Adames", "Matt Chapman", "Jorge Soler", "Tyler Fitzgerald",
        "Grant McCray", "Patrick Bailey", "Thairo Estrada",
        "Heliot Ramos", "Nick Ahmed"
    ],
    "Pittsburgh Pirates": [
        "Oneil Cruz", "Bryan Reynolds", "Andrew McCutchen", "Rowdy Tellez",
        "Endy Rodriguez", "Ji Hwan Bae", "Isiah Kiner-Falefa",
        "Joey Bart", "Michael A. Taylor"
    ],
    "New York Mets": [
        "Francisco Lindor", "Juan Soto", "Mark Vientos", "Pete Alonso",
        "Jesse Winker", "Starling Marte", "Jeff McNeil",
        "Francisco Alvarez", "Tyrone Taylor"
    ],
    "Chicago White Sox": [
        "Luis Robert Jr.", "Andrew Vaughn", "Lenyn Sosa", "Gavin Sheets",
        "Martin Maldonado", "Korey Lee", "Miguel Vargas",
        "Brooks Baldwin", "Zach DeLoach"
    ],
    "Milwaukee Brewers": [
        "Sal Frelick", "Christian Yelich", "William Contreras", "Rhys Hoskins",
        "Joey Wiemer", "Brice Turang", "Andruw Monasterio",
        "Jackson Chourio", "Gary Sanchez"
    ],
    "Washington Nationals": [
        "CJ Abrams", "Dylan Crews", "Nathaniel Lowe", "Keibert Ruiz",
        "Jesse Winker", "Stone Garrett", "Ildemaro Vargas",
        "Jacob Young", "Luis Garcia"
    ],
    "Chicago Cubs": [
        "Nico Hoerner", "Ian Happ", "Seiya Suzuki", "Michael Busch",
        "Cody Bellinger", "Dansby Swanson", "Miles Mastrobuoni",
        "Miguel Amaya", "Pete Crow-Armstrong"
    ],
    "Minnesota Twins": [
        "Carlos Correa", "Byron Buxton", "Ryan Jeffers", "Trevor Larnach",
        "Matt Wallner", "Austin Martin", "Brooks Lee",
        "Christian Vazquez", "Edouard Julien"
    ],
    "Baltimore Orioles": [
        "Gunnar Henderson", "Adley Rutschman", "Pete Alonso", "Taylor Ward",
        "Samuel Basallo", "Tyler O'Neill", "Coby Mayo",
        "Colton Cowser", "Blaze Alexander"
    ],
    "Boston Red Sox": [
        "Roman Anthony", "Trevor Story", "Jarren Duran", "Willson Contreras",
        "Wilyer Abreu", "Caleb Durbin", "Marcelo Mayer",
        "Ceddanne Rafaela", "Carlos Narvaez"
    ],
    "Cincinnati Reds": [
        "TJ Friedl", "Elly De La Cruz", "Tyler Stephenson", "Spencer Steer",
        "Santiago Espinal", "Rece Hinds", "Stuart Fairchild",
        "Luke Maile", "Will Benson"
    ],
    "Tampa Bay Rays": [
        "Yandy Diaz", "Jonathan Aranda", "Junior Caminero", "Ben Williamson",
        "Cedric Mullins", "Jonny DeLuca", "Nick Fortes",
        "Chandler Simpson", "Carson Williams"
    ],
    "St. Louis Cardinals": [
        "Masyn Winn", "Brendan Donovan", "Nolan Arenado", "Alec Burleson",
        "Victor Scott II", "Ivan Herrera", "JJ Wetherholt",
        "Michael Siani", "Nolan Gorman"
    ],
    "Detroit Tigers": [
        "Riley Greene", "Spencer Torkelson", "Parker Meadows", "Justyn-Henry Malloy",
        "Andy Ibanez", "Jace Jung", "Jake Rogers",
        "Zach McKinstry", "Matt Vierling"
    ],
    "San Diego Padres": [
        "Fernando Tatis Jr.", "Manny Machado", "Jackson Merrill", "Kyle Higashioka",
        "Xander Bogaerts", "Jurickson Profar", "Graham Pauley",
        "Eguy Rosario", "Jose Azocar"
    ],
    "Texas Rangers": [
        "Marcus Semien", "Corey Seager", "Adolis Garcia", "Jonah Heim",
        "Wyatt Langford", "Josh Jung", "Leody Taveras",
        "Ezequiel Duran", "Travis Jankowski"
    ],
    "Los Angeles Angels": [
        "Zach Neto", "Mike Trout", "Nolan Schanuel", "Jorge Soler",
        "Yoan Moncada", "Jo Adell", "Josh Lowe",
        "Logan O'Hoppe", "Oswald Peraza"
    ],
    "Houston Astros": [
        "Jose Altuve", "Isaac Paredes", "Yordan Alvarez", "Carlos Correa",
        "Yainer Diaz", "Christian Walker", "Cam Smith",
        "Joey Loperfido", "Jake Meyers"
    ],
    "Philadelphia Phillies": [
        "Kyle Schwarber", "Trea Turner", "Bryce Harper", "Nick Castellanos",
        "Johan Rojas", "Edmundo Sosa", "Bryson Stott",
        "J.T. Realmuto", "Weston Wilson"
    ],
    "Arizona Diamondbacks": [
        "Ketel Marte", "Corbin Carroll", "Christian Walker", "Lourdes Gurriel Jr.",
        "Eugenio Suarez", "Jake McCarthy", "Joc Pederson",
        "Gabriel Moreno", "Kevin Newman"
    ],
    "Los Angeles Dodgers": [
        "Shohei Ohtani", "Freddie Freeman", "Mookie Betts", "Will Smith",
        "Teoscar Hernandez", "Andy Pages", "Miguel Rojas",
        "Gavin Lux", "Dalton Rushing"
    ],
    "Cleveland Guardians": [
        "Steven Kwan", "Chase DeLauter", "Jose Ramirez", "Kyle Manzardo",
        "Rhys Hoskins", "Angel Martinez", "Bo Naylor",
        "Gabriel Arias", "Brayan Rocchio"
    ],
    "Seattle Mariners": [
        "Julio Rodriguez", "Cal Raleigh", "Mitch Haniger", "Luke Raley",
        "Josh Rojas", "Dylan Moore", "Leo Rivas",
        "Tyler Locklear", "Zach Dezenzo"
    ],
    "Oakland Athletics": [
        "Brent Rooker", "Lawrence Butler", "Seth Brown", "JJ Bleday",
        "Shea Langeliers", "Zack Gelof", "Max Schuemann",
        "Abraham Toro", "Daz Cameron"
    ],
    "Toronto Blue Jays": [
        "George Springer", "Addison Barger", "Vladimir Guerrero Jr.", "Alejandro Kirk",
        "Daulton Varsho", "Nathan Lukes", "Kazuma Okamoto",
        "Ernie Clement", "Andres Gimenez"
    ],
    "Colorado Rockies": [
        "Ezequiel Tovar", "Brenton Doyle", "Ryan McMahon", "Charlie Blackmon",
        "Hunter Goodman", "Jake Cave", "Alan Trejo",
        "Drew Romo", "Nolan Jones"
    ],
    "Miami Marlins": [
        "Jazz Chisholm Jr.", "Xavier Edwards", "Josh Bell", "Jonah Bride",
        "Nick Gordon", "Griffin Conine", "Jacob Stallings",
        "Agustin Ramirez", "Otto Lopez"
    ],
    "Kansas City Royals": [
        "Maikel Garcia", "Bobby Witt Jr.", "Vinnie Pasquantino", "Salvador Perez",
        "Isaac Collins", "Jonathan India", "Carter Jensen",
        "Lane Thomas", "Kyle Isbel"
    ],
    "Atlanta Braves": [
        "Ronald Acuna Jr.", "Matt Olson", "Marcell Ozuna", "Austin Riley",
        "Michael Harris II", "Ozzie Albies", "Sean Murphy",
        "Zack Short", "Forrest Wall"
    ],
}

print(f"✓ Loaded lineups for {len(lineups)} teams")

✓ Loaded lineups for 30 teams


In [46]:
# Match each lineup player to their 2025 batting stats
# Score each lineup on wOBA, xwOBA, K%, BB%, HardHit%, Barrel%, wRC+

lineup_scores = []

for team, players in lineups.items():
    team_stats = []
    missing = []
    
    for player in players:
        # Try exact match first
        match = batting_2025[batting_2025['Name'] == player]
        
        # Try last name only if no exact match
        if len(match) == 0:
            last = player.split()[-1]
            match = batting_2025[batting_2025['Name'].str.contains(last, case=False, na=False)]
        
        if len(match) > 0:
            team_stats.append(match.iloc[0])
        else:
            missing.append(player)
    
    if len(team_stats) > 0:
        team_df = pd.DataFrame(team_stats)
        lineup_scores.append({
            'team': team,
            'players_found': len(team_stats),
            'players_missing': len(missing),
            'missing_names': ', '.join(missing),
            'avg_wOBA': team_df['wOBA'].mean(),
            'avg_xwOBA': team_df['xwOBA'].mean(),
            'avg_wRC+': team_df['wRC+'].mean(),
            'avg_HardHit%': team_df['HardHit%'].mean(),
            'avg_Barrel%': team_df['Barrel%'].mean(),
            'avg_K%': team_df['K%'].mean(),
            'avg_BB%': team_df['BB%'].mean(),
        })
    
    print(f"  {team}: {len(team_stats)}/9 matched — missing: {missing if missing else 'none'}")

lineup_df = pd.DataFrame(lineup_scores)
lineup_df.to_csv('../data/lineup_scores.csv', index=False)
print(f"\n✓ Lineup scores built for {len(lineup_df)} teams")

  New York Yankees: 9/9 matched — missing: none
  San Francisco Giants: 7/9 matched — missing: ['Grant McCray', 'Nick Ahmed']
  Pittsburgh Pirates: 9/9 matched — missing: none
  New York Mets: 8/9 matched — missing: ['Jesse Winker']
  Chicago White Sox: 8/9 matched — missing: ['Zach DeLoach']
  Milwaukee Brewers: 8/9 matched — missing: ['Joey Wiemer']
  Washington Nationals: 7/9 matched — missing: ['Jesse Winker', 'Stone Garrett']
  Chicago Cubs: 9/9 matched — missing: none
  Minnesota Twins: 9/9 matched — missing: none
  Baltimore Orioles: 9/9 matched — missing: none
  Boston Red Sox: 9/9 matched — missing: none
  Cincinnati Reds: 6/9 matched — missing: ['Rece Hinds', 'Stuart Fairchild', 'Luke Maile']
  Tampa Bay Rays: 8/9 matched — missing: ['Jonny DeLuca']
  St. Louis Cardinals: 7/9 matched — missing: ['JJ Wetherholt', 'Michael Siani']
  Detroit Tigers: 9/9 matched — missing: none
  San Diego Padres: 8/9 matched — missing: ['Jose Azocar']
  Texas Rangers: 8/9 matched — missing: ['Tr

### Filling in the Gaps

Not every player in a projected lineup had enough plate appearances in 2025 to qualify 
for our main dataset. Some are rookies seeing their first Opening Day. Some are veterans 
who missed time with injuries. Some just did not accumulate enough at bats to clear the 
threshold we set earlier.

For the players we could not match, we have two options. If they had any 2025 MLB data 
at all, even just a handful of games, we can pull those numbers and include them. 
If they have zero 2025 data, we leave them out of the lineup score calculation cleanly 
and note the gap. A lineup score built on 7 real players is more honest than one 
padded with guesses.

This is one of those places where being transparent about your methodology matters 
more than pretending the data is perfect. The data is never perfect. Baseball is not perfect. 
That is part of what makes it beautiful.

In [47]:
# Pull unqualified hitters to catch the missing players
batting_2025_all = batting_stats(2025, 2025, qual=1)

# Check which missing players we can now find
missing_all = [
    'Grant McCray', 'Nick Ahmed', 'Jesse Winker', 'Zach DeLoach',
    'Joey Wiemer', 'Stone Garrett', 'Rece Hinds', 'Stuart Fairchild',
    'Luke Maile', 'Jonny DeLuca', 'Michael Siani', 'Jose Azocar',
    'Travis Jankowski', 'Mitch Haniger', 'Seth Brown', 'Daz Cameron',
    'Charlie Blackmon', 'Jake Cave', 'Alan Trejo', 'Drew Romo',
    'Nick Gordon', 'Zack Short'
]

found = []
still_missing = []
for player in missing_all:
    match = batting_2025_all[batting_2025_all['Name'] == player]
    if len(match) > 0:
        found.append(player)
    else:
        still_missing.append(player)

print(f"Found {len(found)} more players with qual=1:")
for p in found:
    print(f"  ✓ {p}")
print(f"\nStill missing {len(still_missing)} (true rookies/no 2025 data):")
for p in still_missing:
    print(f"  ✗ {p}")

Found 16 more players with qual=1:
  ✓ Grant McCray
  ✓ Nick Ahmed
  ✓ Jesse Winker
  ✓ Joey Wiemer
  ✓ Rece Hinds
  ✓ Stuart Fairchild
  ✓ Luke Maile
  ✓ Jonny DeLuca
  ✓ Michael Siani
  ✓ Jose Azocar
  ✓ Travis Jankowski
  ✓ Seth Brown
  ✓ Daz Cameron
  ✓ Alan Trejo
  ✓ Drew Romo
  ✓ Zack Short

Still missing 6 (true rookies/no 2025 data):
  ✗ Zach DeLoach
  ✗ Stone Garrett
  ✗ Mitch Haniger
  ✗ Charlie Blackmon
  ✗ Jake Cave
  ✗ Nick Gordon


### The Final Six

Six players across all 28 Opening Day lineups have no 2025 MLB data to pull from.

Two are rookies stepping onto an Opening Day roster for the first time. One missed the 
entire 2025 season with injury. One walked away from the game after a long career. 
The other two simply did not play enough to leave a statistical footprint.

None of them are cleanup hitters. None of them change the character of their lineup 
in a meaningful way. We note them, exclude them from the averages, and move on. 
Honesty about data gaps is part of doing this right.

## 📋 Scoring Every Opening Day Lineup

Now we put a number on each lineup. Using the full 2025 batting dataset, including 
players who did not reach the standard qualification threshold, we match every projected 
starter to their real offensive numbers from last season.

Each lineup gets scored across six dimensions: wOBA, xwOBA, wRC+, hard hit rate, 
barrel rate, strikeout rate, and walk rate. The average across all matched players 
becomes that team's Lineup Quality Score.

This is not a perfect measure of how good a team will be in 2026. Rosters change. 
Players grow. Injuries happen. But it is an honest snapshot of the offensive talent 
each manager is penciling in on Opening Day, grounded entirely in real data from 
the season that just ended.

In [48]:
# Rebuild lineup scores using full batting dataset
lineup_scores = []

for team, players in lineups.items():
    team_stats = []
    missing = []
    
    for player in players:
        match = batting_2025_all[batting_2025_all['Name'] == player]
        if len(match) == 0:
            last = player.split()[-1]
            match = batting_2025_all[batting_2025_all['Name'].str.contains(last, case=False, na=False)]
        if len(match) > 0:
            team_stats.append(match.iloc[0])
        else:
            missing.append(player)
    
    if len(team_stats) > 0:
        team_df = pd.DataFrame(team_stats)
        lineup_scores.append({
            'team': team,
            'players_found': len(team_stats),
            'players_missing': len(missing),
            'missing_names': ', '.join(missing),
            'avg_wOBA': team_df['wOBA'].mean(),
            'avg_xwOBA': team_df['xwOBA'].mean(),
            'avg_wRC+': team_df['wRC+'].mean(),
            'avg_HardHit%': team_df['HardHit%'].mean(),
            'avg_Barrel%': team_df['Barrel%'].mean(),
            'avg_K%': team_df['K%'].mean(),
            'avg_BB%': team_df['BB%'].mean(),
        })

lineup_df = pd.DataFrame(lineup_scores)
lineup_df.to_csv('../data/lineup_scores.csv', index=False)

print(f"✓ Final lineup scores for {len(lineup_df)} teams")
print()
print(lineup_df[['team','players_found','avg_wOBA','avg_xwOBA','avg_wRC+']].sort_values('avg_xwOBA', ascending=False).to_string(index=False))

✓ Final lineup scores for 30 teams

                 team  players_found  avg_wOBA  avg_xwOBA   avg_wRC+
     New York Yankees              9  0.353556   0.351111 128.000000
    Toronto Blue Jays              8  0.333625   0.339875 115.000000
        New York Mets              9  0.329889   0.337111 115.000000
         Chicago Cubs              9  0.330778   0.335111 114.444444
  Los Angeles Dodgers              9  0.336889   0.333000 116.333333
     San Diego Padres              9  0.320222   0.332111 107.000000
   Kansas City Royals              9  0.325333   0.329667 106.555556
       Atlanta Braves              9  0.327000   0.326333 109.333333
       Houston Astros              9  0.328444   0.326222 111.222222
    Baltimore Orioles              9  0.309778   0.323778  99.111111
Philadelphia Phillies              9  0.322222   0.323000 104.666667
      Minnesota Twins              9  0.314000   0.320778 100.111111
 Arizona Diamondbacks              9  0.310778   0.319444  98.11111

In [59]:
# Lineup Quality Bar Chart — all Opening Day teams ranked by xwOBA

lineup_plot = lineup_df[['team', 'avg_xwOBA', 'avg_wRC+']].sort_values('avg_xwOBA', ascending=True).copy()

# Color bars by league (AL/NL) — manual mapping
nl_teams = [
    'New York Mets', 'Philadelphia Phillies', 'Atlanta Braves', 'Miami Marlins', 'Washington Nationals',
    'Chicago Cubs', 'Milwaukee Brewers', 'Cincinnati Reds', 'Pittsburgh Pirates', 'St. Louis Cardinals',
    'Los Angeles Dodgers', 'San Diego Padres', 'Arizona Diamondbacks', 'Colorado Rockies', 'San Francisco Giants'
]

lineup_plot['league'] = lineup_plot['team'].apply(lambda x: 'NL' if x in nl_teams else 'AL')
lineup_plot['color'] = lineup_plot['league'].map({'AL': '#3b82f6', 'NL': '#f97316'})

fig2 = go.Figure()

fig2.add_trace(go.Bar(
    x=lineup_plot['avg_xwOBA'],
    y=lineup_plot['team'],
    orientation='h',
    marker=dict(
        color=lineup_plot['color'],
        opacity=0.85,
        line=dict(width=0.5, color='white')
    ),
    hovertemplate='<b>%{y}</b><br>xwOBA: %{x:.3f}<extra></extra>'
))

# League average reference line
fig2.add_vline(
    x=0.315,
    line_dash='dash',
    line_color='#94a3b8',
    line_width=1.5,
    annotation_text='League Avg (.315)',
    annotation_font=dict(color='#1e293b', size=11),
    annotation_position='top right'
)

fig2.update_layout(
    title=dict(
        text='Opening Day 2026 — Lineup Quality by xwOBA<br><sup>Blue = AL, Orange = NL. xwOBA measures true offensive quality, stripping out luck and defense.</sup>',
        font=dict(color='#0f172a', size=18),
        x=0.05
    ),
    paper_bgcolor='white',
    plot_bgcolor='#f8fafc',
    font=dict(color='#1e293b', family='Arial'),
    xaxis=dict(
        title=dict(text='Average Lineup xwOBA', font=dict(color='#1e293b', size=13)),
        gridcolor='#e2e8f0',
        zerolinecolor='#cbd5e1',
        tickfont=dict(color='#1e293b'),
        range=[0.22, 0.37]
    ),
    yaxis=dict(
        tickfont=dict(color='#1e293b', size=11),
        gridcolor='#e2e8f0',
    ),
    height=950,
    width=1400,
    showlegend=False
)

fig2.show()

### What the Lineup Scores Are Telling Us

The Yankees own the most dangerous lineup on paper in this entire Opening Day field. 
A .351 xwOBA and 128 wRC+ means they create more runs per plate appearance than 
almost any team in baseball right now.

The Rockies sit at the bottom with a .259 xwOBA and 45 wRC+. They are also playing 
at Coors Field, the most run-amplifying park in baseball. The worst lineup in this 
field gets a 15% boost just from their home address. That tension is one of the most 
interesting stories in this whole dataset.

## ⚙️ Building the Run Environment Model

This is where everything we have built comes together.

We have the pitchers. We have the lineups. We have the parks. Now we combine them 
into a single number for each game: the projected run environment.

The logic is straightforward. A pitcher with a low xERA is expected to suppress runs. 
A lineup with a high xwOBA is expected to create them. The park factor tilts the 
playing field before a single pitch is thrown. Put those three things together and 
you get a reasonable estimate of how many total runs should score in each Opening Day game.

This is not a scoreboard prediction. Baseball is beautifully, maddeningly random and 
no model captures that. What this gives us is a baseline expectation. A number that 
says, given everything we know about these two pitchers, these two lineups, and this 
ballpark, here is the run environment we are walking into on Opening Day.

The games that deviate wildly from that baseline are the ones worth watching closest.

In [49]:
# Run Environment Model
# Combines: starter quality (xERA) + lineup quality (xwOBA) + park factor
# Output: projected total runs for each Opening Day game

# First merge everything onto the games dataframe
# We need: home starter xERA, away starter xERA, home lineup xwOBA, away lineup xwOBA, park factor

# Build a starter stats lookup
starter_stats = fg_starters[['Name', 'xERA', 'FIP', 'xFIP', 'SIERA', 'IP', 'K%', 'BB%', 'Stuff+', 'Pitching+']].copy()
starter_stats.columns = ['starter', 'xERA', 'FIP', 'xFIP', 'SIERA', 'IP', 'K_pct', 'BB_pct', 'stuff_plus', 'pitching_plus']

# Build lineup lookup
lineup_lookup = lineup_df.set_index('team')[['avg_xwOBA', 'avg_wRC+', 'avg_HardHit%', 'avg_Barrel%']].to_dict('index')

# Merge starter stats onto games
games_model = games_df.copy()

# Home starter stats
games_model = games_model.merge(
    starter_stats.rename(columns=lambda x: f'home_{x}' if x != 'starter' else x),
    left_on='home_starter', right_on='starter', how='left'
).drop('starter', axis=1)

# Away starter stats
games_model = games_model.merge(
    starter_stats.rename(columns=lambda x: f'away_{x}' if x != 'starter' else x),
    left_on='away_starter', right_on='starter', how='left'
).drop('starter', axis=1)

# Add lineup scores
games_model['home_lineup_xwOBA'] = games_model['home_team'].map(lambda t: lineup_lookup.get(t, {}).get('avg_xwOBA', np.nan))
games_model['away_lineup_xwOBA'] = games_model['away_team'].map(lambda t: lineup_lookup.get(t, {}).get('avg_xwOBA', np.nan))
games_model['home_wRC+'] = games_model['home_team'].map(lambda t: lineup_lookup.get(t, {}).get('avg_wRC+', np.nan))
games_model['away_wRC+'] = games_model['away_team'].map(lambda t: lineup_lookup.get(t, {}).get('avg_wRC+', np.nan))

print("✓ All data merged")
print(f"Shape: {games_model.shape}")
print(games_model[['away_team','home_team','away_xERA','home_xERA','away_lineup_xwOBA','home_lineup_xwOBA','park_factor']].to_string(index=False))

✓ All data merged
Shape: (15, 33)
           away_team             home_team  away_xERA  home_xERA  away_lineup_xwOBA  home_lineup_xwOBA  park_factor
    New York Yankees  San Francisco Giants       3.37       3.58           0.351111           0.275000           94
  Pittsburgh Pirates         New York Mets       2.65       3.42           0.299000           0.337111           95
   Chicago White Sox     Milwaukee Brewers       3.98       3.40           0.307875           0.312889          100
Washington Nationals          Chicago Cubs       4.14       3.75           0.299222           0.335111           94
     Minnesota Twins     Baltimore Orioles       3.43       3.40           0.320778           0.323778          107
      Boston Red Sox       Cincinnati Reds       2.88       3.55           0.319444           0.297889          112
  Los Angeles Angels        Houston Astros       4.03       3.15           0.317889           0.326222          100
      Detroit Tigers      San Diego Pa

### The Model Speaks

The numbers are in. Royals vs Braves projects as the highest scoring Opening Day game 
at nearly 13 total runs. Red Sox vs Reds at Great American Ball Park is right behind it, 
which should surprise nobody who has watched a game in Cincinnati.

At the other end, Rays vs Cardinals projects as a quiet 8 run game at Busch Stadium. 
Drew Rasmussen and Matthew Liberatore are both genuinely good pitchers who do not get 
nearly enough attention. That game might be the best pitching matchup nobody is watching 
on Opening Day.

The Tigers vs Padres game returns no projection because the Padres have not named a 
starter yet. When they do, that number will fill in. Tarik Skubal on the mound means 
the under is worth watching regardless of who San Diego throws out there.

In [50]:
# Run Environment Model
# Projected runs scored by each team = f(opponent xERA, own lineup xwOBA, park factor)
# League average xERA ~ 3.80, league average xwOBA ~ .315

LEAGUE_AVG_RUNS = 4.5  # runs per game per team, 2025 MLB average

def project_runs(batting_xwOBA, pitching_xERA, park_factor):
    if pd.isna(batting_xwOBA) or pd.isna(pitching_xERA):
        return np.nan
    # Lineup modifier: how much better/worse than average is this lineup
    lineup_mod = batting_xwOBA / 0.315
    # Pitcher modifier: how much better/worse than average is this pitcher
    pitcher_mod = 3.80 / pitching_xERA
    # Park modifier
    park_mod = park_factor / 100
    return round(LEAGUE_AVG_RUNS * lineup_mod * pitcher_mod * park_mod, 2)

# Project runs for each team in each game
games_model['home_proj_runs'] = games_model.apply(
    lambda r: project_runs(r['home_lineup_xwOBA'], r['away_xERA'], r['park_factor']), axis=1
)
games_model['away_proj_runs'] = games_model.apply(
    lambda r: project_runs(r['away_lineup_xwOBA'], r['home_xERA'], r['park_factor']), axis=1
)

games_model['total_proj_runs'] = games_model['home_proj_runs'] + games_model['away_proj_runs']

# Sort by total projected runs
results = games_model[['away_team','home_team','away_starter','home_starter',
                        'venue','park_factor','away_proj_runs','home_proj_runs',
                        'total_proj_runs']].sort_values('total_proj_runs', ascending=False)

print("Opening Day 2026 — Projected Run Environments")
print("=" * 60)
print(results.to_string(index=False))

Opening Day 2026 — Projected Run Environments
           away_team             home_team    away_starter       home_starter                    venue  park_factor  away_proj_runs  home_proj_runs  total_proj_runs
  Kansas City Royals        Atlanta Braves     Cole Ragans         Chris Sale              Truist Park           99            6.22            6.57            12.79
      Boston Red Sox       Cincinnati Reds Garrett Crochet      Andrew Abbott Great American Ball Park          112            5.47            6.29            11.76
       Texas Rangers Philadelphia Phillies  Nathan Eovaldi Cristopher Sanchez       Citizens Bank Park          105            5.61            6.10            11.71
  Pittsburgh Pirates         New York Mets     Paul Skenes     Freddy Peralta               Citi Field           95            4.51            6.56            11.07
     Minnesota Twins     Baltimore Orioles        Joe Ryan      Trevor Rogers             Camden Yards          107            5.

In [63]:
run_plot2 = games_model[['away_team', 'home_team', 'away_starter', 
                          'home_starter', 'total_proj_runs']].dropna().copy()

run_plot2['matchup'] = run_plot2['away_team'] + ' @ ' + run_plot2['home_team']
run_plot2['label'] = run_plot2['away_starter'] + ' vs ' + run_plot2['home_starter']
run_plot2 = run_plot2.sort_values('total_proj_runs', ascending=True)

fig4b = go.Figure()

fig4b.add_trace(go.Scatter(
    x=run_plot2['total_proj_runs'],
    y=run_plot2['matchup'],
    mode='markers+text',
    text=run_plot2['total_proj_runs'].round(1).astype(str),
    textposition='middle right',
    textfont=dict(size=11, color='#1e293b'),
    marker=dict(
        size=18,
        color=run_plot2['total_proj_runs'],
        colorscale='RdYlGn',
        showscale=True,
        colorbar=dict(title='Total Runs', tickfont=dict(color='#1e293b')),
        line=dict(width=1, color='white')
    ),
    hovertemplate='<b>%{y}</b><br>Projected runs: %{x:.2f}<extra></extra>'
))

fig4b.update_layout(
    title=dict(
        text='Opening Day 2026 — Projected Run Environment<br><sup>Based on starter xERA, lineup xwOBA, and park factors.</sup>',
        font=dict(color='#0f172a', size=18),
        x=0.05
    ),
    paper_bgcolor='white',
    plot_bgcolor='#f8fafc',
    font=dict(color='#1e293b', family='Arial'),
    xaxis=dict(
        title=dict(text='Projected Total Runs', font=dict(color='#1e293b', size=13)),
        gridcolor='#e2e8f0',
        tickfont=dict(color='#1e293b'),
        range=[5, 15]
    ),
    yaxis=dict(
        tickfont=dict(color='#1e293b', size=11),
        gridcolor='#e2e8f0',
    ),
    height=750,
    width=1200
)

fig4b.show()

In [51]:
games_model.to_csv('../data/games_model.csv', index=False)
print("✓ Saved games_model.csv")

✓ Saved games_model.csv


In [52]:
import os
print("Data files saved:")
for f in os.listdir('../data'):
    size = os.path.getsize(f'../data/{f}') / 1024
    print(f"  {f} — {size:.1f} KB")

Data files saved:
  arsenal_summary.csv — 26.9 KB
  fg_stats_2025.csv — 48.7 KB
  games_model.csv — 4.8 KB
  lineup_scores.csv — 4.2 KB
  opening_day_games.csv — 1.9 KB
  starter_pitches_2025.parquet — 11142.7 KB
